# PPO — Mario DRL Playtester

**Projeto:** Avaliação Automática de Dificuldade em Jogos de Plataforma 2D com Deep RL
**Disciplina:** Redes Neurais Artificiais — PPGCC/UNESP

Treina o agente **PPO** em 4 fases (1-1, 1-2, 4-1, 8-1) × 3 seeds (42, 123, 2024)
sob orçamento computacional fixo de **500 000 timesteps** por configuração.

Stack moderna: `gymnasium ≥ 1.0` nativo, `stable-baselines3 ≥ 2.4`,
fork `tooichitake/gymnasium-mario` (nes-py + gym-super-mario-bros compatíveis com gymnasium).
Sem `condacolab`, sem `shimmy`, sem downgrade de Python.

## 1. Instalação

In [ ]:
# --- 1. Instalação ---
# Valida ambiente, instala compilador (cmake p/ nes-py) e a stack RL.
# Pode ser re-executada com segurança: pula etapas já satisfeitas.
#
# IMPORTANTE: se algum pacote for instalado/atualizado NESTA execução, a célula
# automaticamente REINICIA o kernel ao final — sem isso, conflitos entre o
# numpy em memória (carregado pelo Colab antes do pip) e o numpy em disco
# (após o pip) quebram a importação de torch/tensorboard com erros do tipo:
#   AttributeError: module 'numpy._core._multiarray_umath' has no attribute '_blas_supports_fpe'
# Quando o kernel reiniciar, RE-EXECUTE esta célula — na 2ª passagem ela detecta
# que tudo já está instalado, pula direto para a validação e segue.

import sys, os, subprocess, importlib.util, platform, time

print(f"Python: {sys.version.split()[0]} | Plataforma: {platform.system()} {platform.machine()}")
assert sys.version_info >= (3, 10), (
    "Requer Python ≥ 3.10. No Colab default (3.11+) funciona; "
    "em outra máquina suba para 3.10/3.11/3.12."
)

def _sh(cmd: str) -> int:
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print((r.stderr or r.stdout)[-800:])
    return r.returncode

def _has(pkg: str) -> bool:
    return importlib.util.find_spec(pkg) is not None

# Flag para detectar se já tentamos um restart nesta sessão (evita loop infinito).
_RESTART_FLAG = "/tmp/.mario_drl_restart_done"
already_restarted = os.path.exists(_RESTART_FLAG)
installed_new = False

# 1) cmake/g++ para compilar nes-py (binding C++ do emulador NES)
if subprocess.run("which cmake", shell=True, capture_output=True).returncode != 0:
    print("► instalando cmake + build-essential...")
    _sh("apt-get update -qq")
    _sh("apt-get install -y -qq cmake build-essential")
    installed_new = True
print("✓ cmake disponível")

# 2) Clone do fork (sem reinstalar se já existe)
REPO_DIR = "/content/gymnasium-mario" if os.path.isdir("/content") else "/tmp/gymnasium-mario"
if not os.path.isdir(REPO_DIR):
    _sh(f"git clone --depth=1 https://github.com/tooichitake/gymnasium-mario.git {REPO_DIR}")

# 3) Mario forks (gymnasium-native, mantém info dict completo do Kauten)
if not (_has("nes_py") and _has("gym_super_mario_bros")):
    print("► instalando nes-py + gymnasium-super-mario-bros (forks)...")
    _sh(f'pip install --quiet "{REPO_DIR}/nes-py"')
    _sh(f'pip install --quiet "{REPO_DIR}/gymnasium-super-mario-bros"')
    installed_new = True

# 4) RL stack moderna
if not (_has("stable_baselines3") and _has("gymnasium")):
    print("► instalando RL stack...")
    _sh('pip install --quiet "stable-baselines3>=2.4.0" "gymnasium>=1.0.0" "torch>=2.4"')
    installed_new = True

# 5) Análise + visualização
if not (_has("imageio") and _has("seaborn") and _has("cv2")):
    print("► instalando análise + viz...")
    _sh('pip install --quiet "opencv-python" "imageio[ffmpeg]" tensorboard pandas scipy '
        'matplotlib seaborn tqdm rich psutil')
    installed_new = True

# 6) Se algo foi instalado AGORA, restart é mandatório.
if installed_new and not already_restarted:
    open(_RESTART_FLAG, "w").write("1")
    print()
    print("=" * 64)
    print("⚠  PACOTES NOVOS INSTALADOS — REINICIANDO O KERNEL")
    print("   (necessário p/ numpy/torch/tensorboard recarregarem coerentes)")
    print()
    print("   Após o restart automático: RE-EXECUTE ESTA CÉLULA.")
    print("   Na 2ª execução ela vai pular tudo e seguir direto p/ validação.")
    print("=" * 64)
    time.sleep(3)
    os.kill(os.getpid(), 9)   # força restart do kernel no Colab/Jupyter

# 7) Validação (executa só quando nada foi instalado nesta passagem)
print()
import torch, gymnasium as gym
import stable_baselines3 as sb3
import gym_super_mario_bros, nes_py  # noqa: F401
print(f"✓ torch {torch.__version__} | CUDA: {torch.cuda.is_available()}"
      + (f" | GPU: {torch.cuda.get_device_name(0)}" if torch.cuda.is_available() else ""))
print(f"✓ gymnasium {gym.__version__} | stable_baselines3 {sb3.__version__}")
print(f"✓ gym_super_mario_bros / nes_py compilados e importáveis")

# 8) Smoke test do env (1 reset + 1 step)
import gym_super_mario_bros as gsmb
from gym_super_mario_bros.actions import SIMPLE_MOVEMENT
from nes_py.wrappers import JoypadSpace
_env = gsmb.make("SuperMarioBros-1-1-v0", render_mode="rgb_array")
_env = JoypadSpace(_env, SIMPLE_MOVEMENT)
_obs, _info = _env.reset(seed=0)
_obs, _r, _term, _trunc, _info = _env.step(0)
print(f"✓ env smoke test: obs={_obs.shape}, info_keys={sorted(_info)[:8]}...")
_env.close()

# Limpa o flag de restart para próximas re-execuções (idempotência limpa)
try: os.remove(_RESTART_FLAG)
except OSError: pass

print("\n✓✓ Ambiente pronto.")


## 2. Imports

In [ ]:
# --- 2. Imports ---
# stdlib
import os, sys, csv, json, time, shutil, random
from pathlib import Path
from datetime import datetime, timezone

# scientific
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import imageio.v2 as imageio

# RL stack (gymnasium nativo)
import gymnasium as gym
import torch

# Super Mario Bros (fork tooichitake — gymnasium-compatible)
import gym_super_mario_bros
from gym_super_mario_bros.actions import SIMPLE_MOVEMENT
from nes_py.wrappers import JoypadSpace

# Stable-Baselines3
from stable_baselines3 import DQN, PPO, A2C
from stable_baselines3.common.callbacks import BaseCallback, CheckpointCallback
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.atari_wrappers import MaxAndSkipEnv, WarpFrame
from stable_baselines3.common.vec_env import (
    DummyVecEnv, SubprocVecEnv, VecFrameStack, VecTransposeImage
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✓ Imports OK | device={DEVICE}")


## 3. Storage (Colab Drive ou local)

In [ ]:
# --- 3. Storage (Colab Drive ou pasta local) ---
# Em Colab: monta o Drive para persistir entre sessões (sessão Colab cai em ~12h).
# Fora do Colab: pasta local ./mario_drl_results.

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    try:
        from google.colab import drive
        if not os.path.ismount("/content/drive"):
            drive.mount("/content/drive", force_remount=False)
        BASE_DIR = Path("/content/drive/MyDrive/mario_drl_results")
        print(f"✓ Colab + Drive montado em {BASE_DIR}")
    except Exception as e:
        print(f"✗ falha ao montar Drive: {e}")
        BASE_DIR = Path("/content/mario_drl_results")
        print(f"  → caindo para pasta local efêmera: {BASE_DIR}")
else:
    BASE_DIR = Path("./mario_drl_results").resolve()
    print(f"✓ Local: {BASE_DIR}")

# Estrutura:
#  BASE_DIR/
#   ├── models/{algo}/{algo}_{stage}_{seed}.zip
#   ├── models/checkpoints/{algo}/{algo}_{stage}_{seed}/{algo}_{stage}_{seed}_{N}_steps.zip
#   ├── logs/{algo}/{algo}_{stage}_{seed}_eval.csv
#   ├── logs/tb/{algo}/{tag}/         (TensorBoard)
#   ├── figures/                       (PNG, PDF — gerados na análise)
#   └── videos/                        (GIFs — verificação visual)
MODELS_DIR = BASE_DIR / "models"
LOGS_DIR   = BASE_DIR / "logs"
FIGS_DIR   = BASE_DIR / "figures"
VIDS_DIR   = BASE_DIR / "videos"
for d in [MODELS_DIR, LOGS_DIR, FIGS_DIR, VIDS_DIR]:
    d.mkdir(parents=True, exist_ok=True)
print(f"✓ Pastas prontas em {BASE_DIR}")


## 4. Configuração global

In [ ]:
# --- 4. Configuração global ---
# SMOKE_TEST=True valida o pipeline em ~3-5 minutos antes de disparar o experimento real.
# Sempre rode com SMOKE_TEST=True na primeira execução em ambiente novo.

SMOKE_TEST = False   # ⚠️ True = 10k timesteps, 1 fase, 1 seed (sanity check)

# Protocolo experimental (Tabela 1 do paper)
STAGES         = ["1-1", "1-2", "4-1", "8-1"]
SEEDS          = [42, 123, 2024]
ALGOS          = ["DQN", "PPO", "A2C"]
TOTAL_TIMESTEPS = 500_000
EVAL_FREQ       = 10_000        # avaliação a cada N timesteps de treino
N_EVAL_EPISODES = 5             # 5 episódios determinísticos por checkpoint
N_CHECKPOINTS   = 5             # 5 checkpoints uniformes por treino

# Comprimentos canônicos das fases (em pixels) — usados na normalização da
# distância (Métrica Grupo II: d_bar = max_x_pos / stage_length)
STAGE_LENGTH = {"1-1": 3266, "1-2": 3266, "4-1": 3866, "8-1": 3266}

if SMOKE_TEST:
    print("⚠️  SMOKE_TEST ativo — reduzindo escopo.")
    STAGES_TO_RUN    = ["1-1"]
    SEEDS_TO_RUN     = [42]
    TOTAL_TIMESTEPS  = 10_000
    EVAL_FREQ        = 2_500
    N_EVAL_EPISODES  = 2
    N_CHECKPOINTS    = 2
else:
    STAGES_TO_RUN    = STAGES
    SEEDS_TO_RUN     = SEEDS

print(f"Configuração: {len(STAGES_TO_RUN)} fases × {len(SEEDS_TO_RUN)} seeds × "
      f"{TOTAL_TIMESTEPS:,} timesteps | eval a cada {EVAL_FREQ:,} ({N_EVAL_EPISODES} ep det.)")


## 5. Hiperparâmetros (PPO)

In [ ]:
# --- 5. Hiperparâmetros (PPO) ---
# Tabela 2 do paper. Apenas o algoritmo deste notebook é exposto.

HPARAMS = {
    "PPO": dict(
        learning_rate  = 2.5e-4,
        n_steps        = 128,
        batch_size     = 256,
        n_epochs       = 4,
        gamma          = 0.99,
        gae_lambda     = 0.95,
        clip_range     = 0.1,
        ent_coef       = 0.01,
        vf_coef        = 0.5,
        max_grad_norm  = 0.5,
        n_envs         = 8,   # PPO on-policy: paralelismo via SubprocVecEnv
    ),
}
ALGO_NAME = "PPO"

print(f"✓ Hiperparâmetros carregados: {ALGO_NAME}")
for k, v in HPARAMS[ALGO_NAME].items():
    print(f"  {k:<24s} = {v}")


## 6. Environment factory

In [ ]:
# --- 6. Environment factory ---
# Pipeline de pré-processamento equivalente ao clássico Atari + Mario:
#   make() → JoypadSpace(SIMPLE_MOVEMENT) → MaxAndSkipEnv(4) → WarpFrame(84,84) → Monitor
#   → VecEnv (Dummy ou Subproc) → VecFrameStack(4) → VecTransposeImage (CHW)
#
# Observação final: (4, 84, 84) uint8 — input do CnnPolicy da SB3.

def set_global_seed(seed: int) -> None:
    """Fixa todas as fontes de aleatoriedade (numpy, random, torch, cuda)."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_mario_env(stage: str, seed: int = 0, render_mode: str = "rgb_array"):
    """Factory para UM env Mario (não vetorizado).

    Retorna a thunk (closure) para uso com SubprocVecEnv/DummyVecEnv —
    a construção real acontece dentro do worker process.
    """
    def _thunk():
        # Imports DENTRO do thunk: SubprocVecEnv usa cloudpickle, mas em alguns
        # cenários (spawn em vez de fork, ou Windows) o subprocess pode não
        # herdar os imports do notebook. Re-importar aqui é defensivo.
        import gym_super_mario_bros as _gsmb
        from gym_super_mario_bros.actions import SIMPLE_MOVEMENT as _SM
        from nes_py.wrappers import JoypadSpace as _JS
        from stable_baselines3.common.atari_wrappers import (
            MaxAndSkipEnv as _MS, WarpFrame as _WF)
        from stable_baselines3.common.monitor import Monitor as _Mon
        env_id = f"SuperMarioBros-{stage}-v0"
        env = _gsmb.make(env_id, render_mode=render_mode)
        env = _JS(env, _SM)                # 256 → 7 ações discretas
        env = _MS(env, skip=4)             # frame skip + max-pool
        env = _WF(env, width=84, height=84)  # grayscale 84×84
        env = _Mon(env)                    # episode statistics
        return env
    return _thunk


def make_vec_env_mario(stage: str, n_envs: int = 1, seed: int = 0,
                       use_subproc: bool | None = None):
    """Cria VecEnv com n cópias + FrameStack(4) + Transpose para CHW.

    Para n_envs=1 usa DummyVecEnv (sem overhead de subprocess).
    Para n_envs>1 usa SubprocVecEnv (paralelismo real via fork).
    """
    if use_subproc is None:
        use_subproc = n_envs > 1
    thunks = [make_mario_env(stage, seed=seed + i) for i in range(n_envs)]
    VecCls = SubprocVecEnv if use_subproc else DummyVecEnv
    venv = VecCls(thunks)
    venv = VecFrameStack(venv, n_stack=4, channels_order="last")
    venv = VecTransposeImage(venv)
    venv.seed(seed)
    return venv


# Sanity check rápido (dimensão da observação)
_smoke = make_vec_env_mario(STAGES_TO_RUN[0], n_envs=1, seed=0)
_obs = _smoke.reset()
print(f"✓ Env factory OK | obs shape: {_obs.shape} (esperado: (1, 4, 84, 84))")
assert _obs.shape == (1, 4, 84, 84), f"Pipeline quebrado: obs={_obs.shape}"
_smoke.close()


## 7. Callback de avaliação

In [ ]:
# --- 7. Callback de avaliação (Grupo I + Grupo II) ---
# A cada EVAL_FREQ timesteps de treino, roda N_EVAL_EPISODES episódios
# determinísticos no eval_env (seed descorrelacionada do treino) e escreve
# UMA LINHA POR EPISÓDIO no CSV — persistência incremental sobrevive a crash.

class MarioEvalCallback(BaseCallback):
    """Avalia determinístico e persiste métricas Grupo I + Grupo II em CSV."""

    HEADER = [
        "algo", "stage", "seed",
        "timestep", "ep_idx",
        "ep_reward", "ep_length",
        "x_pos_max", "flag_get",
        "deaths", "time_remaining",
        "ts_iso",
    ]

    def __init__(self, eval_env, *, eval_freq: int, n_eval_episodes: int,
                 log_path: str | Path, algo: str, stage: str, seed: int,
                 stage_length: int, verbose: int = 0):
        super().__init__(verbose)
        self.eval_env        = eval_env
        self.eval_freq       = max(int(eval_freq), 1)
        self.n_eval_episodes = int(n_eval_episodes)
        self.log_path        = Path(log_path)
        self.algo, self.stage, self.seed = algo, stage, seed
        self.stage_length    = int(stage_length)
        self._last_eval      = 0
        self._init_log()

    def _init_log(self) -> None:
        self.log_path.parent.mkdir(parents=True, exist_ok=True)
        if not self.log_path.exists():
            with open(self.log_path, "w", newline="") as f:
                csv.writer(f).writerow(self.HEADER)

    def _on_step(self) -> bool:
        # num_timesteps cresce em n_envs por step do worker — comparação contra
        # o valor absoluto garante o intervalo solicitado.
        if self.num_timesteps - self._last_eval >= self.eval_freq:
            self._last_eval = self.num_timesteps
            self._evaluate()
        return True

    def _evaluate(self) -> None:
        rows = []
        for ep_idx in range(self.n_eval_episodes):
            obs = self.eval_env.reset()
            done = np.array([False])
            ep_r, ep_len = 0.0, 0
            x_max, flag, deaths, time_rem = 0, False, 0, 0
            prev_life = None
            while not done.any():
                action, _ = self.model.predict(obs, deterministic=True)
                obs, reward, done, infos = self.eval_env.step(action)
                ep_r  += float(reward[0])
                ep_len += 1
                info = infos[0] if isinstance(infos, (list, tuple)) else infos
                x_max = max(x_max, int(info.get("x_pos", 0)))
                if info.get("flag_get", False):
                    flag = True
                cur_life = info.get("life", None)
                if (prev_life is not None and cur_life is not None
                        and cur_life < prev_life):
                    deaths += 1
                prev_life = cur_life
                time_rem = int(info.get("time", 0))
            rows.append([
                self.algo, self.stage, self.seed,
                int(self.num_timesteps), ep_idx,
                round(ep_r, 4), int(ep_len),
                int(x_max), bool(flag),
                int(deaths), int(time_rem),
                datetime.now(timezone.utc).isoformat(timespec="seconds"),
            ])

        with open(self.log_path, "a", newline="") as f:
            csv.writer(f).writerows(rows)

        med_r = float(np.median([r[5] for r in rows]))
        compl = sum(1 for r in rows if r[8]) / max(self.n_eval_episodes, 1)
        d_bar = float(np.median([r[7] for r in rows])) / self.stage_length
        print(f"  [{self.algo}/{self.stage}/seed={self.seed}] "
              f"ts={self.num_timesteps:>7,} | med_r={med_r:6.1f} | "
              f"d̄={d_bar:.2f} | τ={compl:.0%}")

print("✓ MarioEvalCallback definido")


## 8. Função de treinamento

In [ ]:
# --- 8. Função de treinamento ---
# train_one(algo, stage, seed):
#   - idempotente: se o modelo final existe, pula
#   - resume from checkpoint: detecta o último checkpoint e continua o treino
#     a partir dele com reset_num_timesteps=False (essencial p/ Colab, que
#     desconecta a sessão em ~12h)
#   - persiste 5 checkpoints uniformes em MODELS_DIR/checkpoints/{algo}/{tag}/
#   - escreve CSV de avaliação incrementalmente (sobrevive a crash)

_ALGO_CLS = {"DQN": DQN, "PPO": PPO, "A2C": A2C}


def _find_latest_ckpt(ckpt_dir: Path, tag: str):
    """Retorna (path, completed_steps) do checkpoint mais recente, ou (None, 0)."""
    ckpts = list(ckpt_dir.glob(f"{tag}_*_steps.zip"))
    if not ckpts:
        return None, 0
    # nome: {tag}_{N}_steps.zip → extrai N
    def _n(p):
        try:
            return int(p.stem.split("_")[-2])
        except (ValueError, IndexError):
            return -1
    ckpts = [p for p in ckpts if _n(p) > 0]
    if not ckpts:
        return None, 0
    latest = max(ckpts, key=_n)
    return latest, _n(latest)


def train_one(algo: str, stage: str, seed: int,
              total_timesteps: int = None,
              eval_freq: int = None,
              n_eval_episodes: int = None) -> dict:
    """Treina UMA configuração (algo, stage, seed) com idempotência + resume."""
    total_timesteps  = total_timesteps  or TOTAL_TIMESTEPS
    eval_freq        = eval_freq        or EVAL_FREQ
    n_eval_episodes  = n_eval_episodes  or N_EVAL_EPISODES

    tag        = f"{algo}_{stage}_{seed}"
    final_path = MODELS_DIR / algo / f"{tag}.zip"
    ckpt_dir   = MODELS_DIR / "checkpoints" / algo / tag
    eval_log   = LOGS_DIR / algo / f"{tag}_eval.csv"
    final_path.parent.mkdir(parents=True, exist_ok=True)
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    if final_path.exists():
        print(f"✓ [{tag}] modelo final já existe — pulando.")
        return {"status": "skipped", "tag": tag, "path": str(final_path)}

    set_global_seed(seed)
    n_envs   = HPARAMS[algo].get("n_envs", 1)
    hp       = {k: v for k, v in HPARAMS[algo].items() if k != "n_envs"}
    ModelCls = _ALGO_CLS[algo]

    train_env = make_vec_env_mario(stage, n_envs=n_envs, seed=seed)
    eval_env  = make_vec_env_mario(stage, n_envs=1, seed=seed + 9999)

    try:
        # --- Resume ou início ---
        latest_ckpt, completed = _find_latest_ckpt(ckpt_dir, tag)
        if latest_ckpt is not None:
            remaining = total_timesteps - completed
            if remaining <= 0:
                print(f"✓ [{tag}] checkpoint cobre o total — promovendo a final.")
                shutil.copy(latest_ckpt, final_path)
                return {"status": "promoted", "tag": tag, "path": str(final_path)}
            print(f"↻ [{tag}] retomando de {completed:,} ts → faltam {remaining:,}")
            model = ModelCls.load(latest_ckpt, env=train_env, device=DEVICE)
            reset_steps = False
        else:
            remaining   = total_timesteps
            reset_steps = True
            print(f"► [{tag}] iniciando do zero ({total_timesteps:,} timesteps)")
            model = ModelCls(
                "CnnPolicy", train_env,
                seed=seed, verbose=0, device=DEVICE,
                tensorboard_log=str(LOGS_DIR / "tb" / algo),
                **hp,
            )

        # --- Callbacks ---
        # save_freq é contado por iteração do learn loop (= por step do worker
        # principal): em vec_env multi-worker, total_timesteps reais por
        # iteração = n_envs. Para 5 checkpoints uniformes sobre TODO o treino:
        save_freq = max((total_timesteps // N_CHECKPOINTS) // n_envs, 1)
        cb_ckpt = CheckpointCallback(
            save_freq=save_freq, save_path=str(ckpt_dir), name_prefix=tag,
            save_replay_buffer=False, save_vecnormalize=False,
        )
        cb_eval = MarioEvalCallback(
            eval_env, eval_freq=max(eval_freq // n_envs, 1),
            n_eval_episodes=n_eval_episodes, log_path=eval_log,
            algo=algo, stage=stage, seed=seed,
            stage_length=STAGE_LENGTH[stage],
        )

        # --- Treino ---
        t0 = time.time()
        model.learn(
            total_timesteps    = remaining,
            callback           = [cb_ckpt, cb_eval],
            reset_num_timesteps= reset_steps,
            tb_log_name        = tag,
            progress_bar       = True,
        )
        elapsed = time.time() - t0

        model.save(final_path)
        print(f"✓ [{tag}] concluído em {elapsed/60:.1f} min → {final_path.name}")
        return {"status": "completed", "tag": tag, "path": str(final_path),
                "elapsed_sec": elapsed}

    finally:
        try: train_env.close()
        except Exception: pass
        try: eval_env.close()
        except Exception: pass

print("✓ train_one() definido (idempotente + resume from checkpoint)")


## 9. Loop de treinamento (PPO)

In [ ]:
# --- 9. Loop de treinamento (PPO) ---
# Roda TODAS as combinações (stage, seed) para o algoritmo deste notebook.
# Cada run é independente e idempotente — se a sessão Colab cair, basta
# re-executar esta célula que ela retoma do último checkpoint.
#
# Tempo estimado em T4 (Colab Pro):
#   DQN  (n_envs=1):  ~120 min × 12 runs = ~24h
#   PPO  (n_envs=8):  ~50  min × 12 runs = ~10h
#   A2C  (n_envs=16): ~35  min × 12 runs = ~7h

def run_ppo_experiments():
    """Executa as 12 combinações de PPO ({4 fases} × {3 seeds})."""
    results, total = [], len(STAGES_TO_RUN) * len(SEEDS_TO_RUN)
    i = 0
    for stage in STAGES_TO_RUN:
        for seed in SEEDS_TO_RUN:
            i += 1
            print(f"\n{'='*70}")
            print(f"[{i}/{total}] PPO | stage {stage} | seed {seed}")
            print('='*70)
            try:
                res = train_one("PPO", stage, seed)
            except Exception as e:
                res = {"status": "failed", "tag": f"PPO_{stage}_{seed}",
                       "error": str(e)}
                print(f"✗ FALHA [{res['tag']}]: {e}")
            results.append(res)
    print(f"\n{'='*70}\nResumo PPO:")
    for r in results:
        print(f"  {r['status']:>10s} | {r['tag']}")
    return results

# ▶︎ Descomente a linha abaixo para disparar o treino completo.
#   (Mantenha SMOKE_TEST=True na primeira execução p/ validar o pipeline.)
# results = run_ppo_experiments()


## 10. Verificação visual

In [ ]:
# --- 10. Verificação visual ---
# Renderiza o agente treinado jogando uma fase e salva GIF.
#
# CRÍTICO: env.render() em emuladores NES retorna referência ao buffer
# interno. Sem .copy() em cada frame, todas as entradas da lista apontam
# para o mesmo bloco de memória → GIF vira frame estático.

def render_agent_episode(model, stage: str, *,
                         max_steps: int = 2000, fps: int = 15,
                         seed: int = 999, save_path: str | None = None) -> list:
    """Roda 1 episódio determinístico, captura frames RGB e (opcional) salva GIF.

    Retorna a lista de frames para uso em matplotlib.FuncAnimation.to_jshtml().
    """
    # Guardamos referência à camada ANTES do WarpFrame para render() retornar
    # RGB completo (não a versão 84×84 grayscale).
    raw_holder = {}
    def _thunk():
        env = gym_super_mario_bros.make(
            f"SuperMarioBros-{stage}-v0", render_mode="rgb_array")
        env = JoypadSpace(env, SIMPLE_MOVEMENT)
        env = MaxAndSkipEnv(env, skip=4)
        raw_holder["env"] = env   # referência para frames RGB
        env = WarpFrame(env, width=84, height=84)
        env = Monitor(env)
        return env

    venv = DummyVecEnv([_thunk])
    venv = VecFrameStack(venv, n_stack=4, channels_order="last")
    venv = VecTransposeImage(venv)
    venv.seed(seed)
    obs = venv.reset()
    raw = raw_holder["env"]

    frames, x_max, flag, n_steps = [], 0, False, 0
    for _ in range(max_steps):
        frame = raw.render()
        if frame is not None:
            frames.append(frame.copy())   # ← evita aliasing do buffer NES
        action, _ = model.predict(obs, deterministic=True)
        obs, _r, done, infos = venv.step(action)
        info = infos[0]
        x_max = max(x_max, int(info.get("x_pos", 0)))
        if info.get("flag_get", False):
            flag = True
        n_steps += 1
        if done[0]:
            f = raw.render()
            if f is not None:
                frames.append(f.copy())
            break

    venv.close()
    print(f"✓ episódio: {n_steps} passos | max_x={x_max} | flag={flag} | "
          f"{len(frames)} frames")

    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        # duration em MILISSEGUNDOS (não segundos — bug histórico já corrigido)
        imageio.mimsave(save_path, frames,
                        duration=int(round(1000 / fps)), loop=0)
        print(f"✓ GIF salvo: {save_path}")

    return frames


def show_frames_inline(frames, fps: int = 15, title: str = ""):
    """Anima frames inline no Jupyter via FuncAnimation.to_jshtml().
    Funciona em VSCode, JupyterLab, Colab — diferente de Image(filename=GIF)
    que em alguns clientes mostra só o 1º frame.
    """
    from matplotlib import animation
    from IPython.display import HTML
    if not frames:
        print("(sem frames)"); return
    fig, ax = plt.subplots(figsize=(5, 4.5))
    im = ax.imshow(frames[0]); ax.axis("off")
    if title: ax.set_title(title)
    def _upd(i):
        im.set_data(frames[i])
        return [im]
    anim = animation.FuncAnimation(fig, _upd, frames=len(frames),
                                   interval=1000/fps, blit=True)
    plt.close(fig)
    return HTML(anim.to_jshtml())


# Exemplo (descomente após treinar):
# tag = "{ALGO_NAME}_1-1_42"
# model = _ALGO_CLS["{ALGO_NAME}"].load(MODELS_DIR / "{ALGO_NAME}" / f"{tag}.zip")
# frames = render_agent_episode(model, "1-1", save_path=VIDS_DIR / f"{tag}.gif")
# show_frames_inline(frames, title=tag)


## Próximos passos

1. **Rodar smoke test** — mantenha `SMOKE_TEST = True` na célula 9 e execute o
   notebook inteiro. Tempo total: ~3-5 min. Confirma que o pipeline está OK.

2. **Disparar treino completo** — coloque `SMOKE_TEST = False`, descomente a
   linha `results = run_ppo_experiments()` na célula 19 e rode.

3. **Se a sessão Colab cair** — basta reabrir o notebook, executar de novo a
   partir da célula 1 (Drive remonta, instalações são pulam, modelos finais
   já existem). O loop só vai retomar os runs incompletos, do último
   checkpoint salvo.

4. **Verificação visual** — célula 21 carrega o modelo de uma config específica
   e gera um GIF do agente jogando. Use para conferir qualitativamente se o
   agente aprendeu (deve avançar para a direita em vez de pular no mesmo lugar).

5. **Rodar os outros algoritmos** — abra 01_train_dqn.ipynb e 03_train_a2c.ipynb e repita.

6. **Análise** — quando os 36 runs (3 algoritmos × 12 cada) estiverem completos,
   abra `04_analysis.ipynb` para computar métricas Grupo I/II, testes
   estatísticos e gerar todas as figuras do paper.